In [1]:
import os

from google import genai

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_community.retrievers import BM25Retriever

from langchain_classic.retrievers import EnsembleRetriever

C:\Users\Dell\AppData\Local\Temp\ipykernel_5136\2199016607.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


**Loads PDFs**

In [2]:
pdf_folder = "day16_pdfs"

documents = []

for file in os.listdir(pdf_folder):

    if file.endswith(".pdf"):

        loader = PyPDFLoader(os.path.join(pdf_folder, file))

        documents.extend(loader.load())

print("Total Pages:", len(documents))

Total Pages: 121


**Split Documents**

In [3]:
splitter = RecursiveCharacterTextSplitter(

    chunk_size=1000,

    chunk_overlap=200

)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 438


**Create Embeddings**

In [4]:
embeddings = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

**Create FAISS database**

In [5]:
vector_db = FAISS.from_documents(

    chunks,

    embeddings

)

print("FAISS Vector DB Created!")

FAISS Vector DB Created!


**Create FAISS Retriever**

In [6]:
vector_retriever = vector_db.as_retriever(

    search_kwargs={"k":3}

)

print("FAISS Retriever Created!")

FAISS Retriever Created!


**Create BM25 Retriever**

In [7]:
bm25 = BM25Retriever.from_documents(chunks)

bm25.k = 3

print("BM25 Retriever Created!")

BM25 Retriever Created!


**Create Ensemble Retriever**

In [8]:
ensemble_retriever = EnsembleRetriever(

    retrievers=[

        bm25,

        vector_retriever

    ],

    weights=[

        0.5,

        0.5

    ]

)

print("Hybrid Retriever Created!")

Hybrid Retriever Created!


**Test BM25 retriever**

In [9]:
query = "What is BERT?"

docs = bm25.invoke(query)

for i, doc in enumerate(docs,1):

    print(f"\nResult {i}\n")

    print(doc.page_content)


Result 1

Gemini: A Family of Highly Capable Multimodal Models
10.4.4. Image understanding and reasoning
Prompt
Look at this sequence of three shapes. What shape should come as the fourth shape? Explain
your reasoning with detailed descriptions of the first shapes.
Model Response
The fourth shape should be a hexagon. The first shape is a triangle, the second shape is a
square, and the third shape is a pentagon. The number of sides in each shape is increasing
by one
Figure 13| Solving a multimodal reasoning problem.The model is able to recognize shapes in the
image, understand their properties and reason about the relationship between them to predict the
next object.
Source: photo taken by an author from the Gemini team.
80

Result 2

response rating by humans.
Translatability Filtering: Not all prompt-response pairs make sense when automatically translated,
and may require expensive localization instead. Example prompts of this type (responses omitted for
space) include:
• (strict wor

**Test FAISS**

In [10]:
docs = vector_retriever.invoke(query)

for i, doc in enumerate(docs,1):

    print(f"\nResult {i}\n")

    print(doc.page_content)


Result 1

ing pre-training, the model is trained on unlabeled
data over different pre-training tasks. For ﬁne-
tuning, the BERT model is ﬁrst initialized with
the pre-trained parameters, and all of the param-
eters are ﬁne-tuned using labeled data from the
downstream tasks. Each downstream task has sep-
arate ﬁne-tuned models, even though they are ini-
tialized with the same pre-trained parameters. The
question-answering example in Figure 1 will serve
as a running example for this section.
A distinctive feature of BERT is its uniﬁed ar-
chitecture across different tasks. There is mini-
mal difference between the pre-trained architec-
ture and the ﬁnal downstream architecture.
Model Architecture BERT’s model architec-
ture is a multi-layer bidirectional Transformer en-
coder based on the original implementation de-
scribed in Vaswani et al. (2017) and released in
the tensor2tensor library.1 Because the use
of Transformers has become common and our im-

Result 2

BERT (Ours)
Trm Trm Trm

**Test Hybrid search**

In [11]:
docs = ensemble_retriever.invoke(query)

for i, doc in enumerate(docs,1):

    print(f"\nResult {i}\n")

    print(doc.page_content)


Result 1

Gemini: A Family of Highly Capable Multimodal Models
10.4.4. Image understanding and reasoning
Prompt
Look at this sequence of three shapes. What shape should come as the fourth shape? Explain
your reasoning with detailed descriptions of the first shapes.
Model Response
The fourth shape should be a hexagon. The first shape is a triangle, the second shape is a
square, and the third shape is a pentagon. The number of sides in each shape is increasing
by one
Figure 13| Solving a multimodal reasoning problem.The model is able to recognize shapes in the
image, understand their properties and reason about the relationship between them to predict the
next object.
Source: photo taken by an author from the Gemini team.
80

Result 2

ing pre-training, the model is trained on unlabeled
data over different pre-training tasks. For ﬁne-
tuning, the BERT model is ﬁrst initialized with
the pre-trained parameters, and all of the param-
eters are ﬁne-tuned using labeled data from the
downstre

**Build context**

In [12]:
context = "\n\n".join(

    [doc.page_content for doc in docs]

)

print(context)

Gemini: A Family of Highly Capable Multimodal Models
10.4.4. Image understanding and reasoning
Prompt
Look at this sequence of three shapes. What shape should come as the fourth shape? Explain
your reasoning with detailed descriptions of the first shapes.
Model Response
The fourth shape should be a hexagon. The first shape is a triangle, the second shape is a
square, and the third shape is a pentagon. The number of sides in each shape is increasing
by one
Figure 13| Solving a multimodal reasoning problem.The model is able to recognize shapes in the
image, understand their properties and reason about the relationship between them to predict the
next object.
Source: photo taken by an author from the Gemini team.
80

ing pre-training, the model is trained on unlabeled
data over different pre-training tasks. For ﬁne-
tuning, the BERT model is ﬁrst initialized with
the pre-trained parameters, and all of the param-
eters are ﬁne-tuned using labeled data from the
downstream tasks. Each downst

**Create prompt**

In [13]:
prompt = f"""
You are an AI assistant.

Answer ONLY using the context below.

If the answer is not available, reply:

"I don't know based on the provided context."

Keep the answer concise (2–4 sentences) and answer only what is asked.

Context:
{context}

Question:
{query}

Answer:
"""

print(prompt)


You are an AI assistant.

Answer ONLY using the context below.

If the answer is not available, reply:

"I don't know based on the provided context."

Keep the answer concise (2–4 sentences) and answer only what is asked.

Context:
Gemini: A Family of Highly Capable Multimodal Models
10.4.4. Image understanding and reasoning
Prompt
Look at this sequence of three shapes. What shape should come as the fourth shape? Explain
your reasoning with detailed descriptions of the first shapes.
Model Response
The fourth shape should be a hexagon. The first shape is a triangle, the second shape is a
square, and the third shape is a pentagon. The number of sides in each shape is increasing
by one
Figure 13| Solving a multimodal reasoning problem.The model is able to recognize shapes in the
image, understand their properties and reason about the relationship between them to predict the
next object.
Source: photo taken by an author from the Gemini team.
80

ing pre-training, the model is trained on u

**Gemini API**

In [14]:
client = genai.Client(
    api_key="Your_API_Key"
)

response = client.models.generate_content(

    model="gemini-2.5-flash",

    contents=prompt

)

print(response.text)

BERT's model architecture is a multi-layer bidirectional Transformer encoder. It is a fine-tuning approach that uses a unified architecture across different tasks. BERT representations are jointly conditioned on both left and right context in all layers.


**Create chatbot function**

In [15]:
def ask_question(question):

    docs = ensemble_retriever.invoke(question)

    context = "\n\n".join(

        [doc.page_content for doc in docs]

    )

    prompt = f"""
    You are an AI assistant.

Rules:
1. Answer ONLY using the provided context.
2. If the answer is not found in the context, reply exactly:
   "I don't know based on the provided context."
3. Keep the answer short and concise (maximum 2-3 sentences).
4. Do NOT copy long paragraphs from the context.
5. Answer only the user's question.
6. If the question asks "Who", return only the person's or organization's name.
7. If the question asks "What", provide only the definition or explanation.
8. If the question asks "When", provide only the date or time.
9. If the question asks "Where", provide only the location.
10. Do not include extra details unless the user specifically asks for them.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    response = client.models.generate_content(

        model="gemini-2.5-flash",

        contents=prompt

    )

    return response.text

**test function**

In [16]:
print(ask_question("What is BERT?"))

print()

print(ask_question("Who developed Gemini?"))

print()

print(ask_question("Explain Transformer architecture."))

BERT is a model architecture that is a multi-layer bidirectional Transformer encoder. It uses a bidirectional Transformer and has a unified architecture across different tasks. BERT is also a fine-tuning approach for pre-trained parameters.

Google

The Transformer architecture uses stacked self-attention and point-wise, fully connected layers for both its encoder and decoder. The encoder consists of a stack of identical layers, each with a multi-head self-attention mechanism and a position-wise fully connected feed-forward network. The model employs residual connections around sub-layers, followed by layer normalization.


**ChatBot**

In [17]:
while True:

    question = input("You: ")

    if question.lower() == "exit":

        print("Goodbye!")

        break

    answer = ask_question(question)

    print("\nAssistant:\n")

    print(answer)

    print("-"*60)

You:  How transformer works?



Assistant:

The Transformer uses multi-head attention in three ways, following an architecture of stacked self-attention and point-wise, fully connected layers for both the encoder and decoder. The encoder contains self-attention layers, and the decoder has self-attention layers and encoder-decoder attention layers. The encoder's layers each feature a multi-head self-attention mechanism and a position-wise fully connected feed-forward network.
------------------------------------------------------------


You:  exit


Goodbye!
